# 02 – FBP Baseline (Baselines / Evaluation – Sophia)

**Question:** How good is plain Filtered Back-Projection (FBP) when only a few projection angles are used?

**Approach**
1. Load the SinoCT sinograms (`.npy`, 984 angles × 888 detector pixels per slice).
2. Ground truth = FBP with **all 984 views** (computed once, cached on Drive).
3. Sparse-view FBP = keep only $n$ evenly spaced angles (e.g. $n = 60$), see `view_indices`.
4. Compare sparse FBP vs. ground truth with PSNR, SSIM and RMSE (`metrics.py`),
   for the whole image and inside an automatic head mask.
5. Save all results to a CSV and plot quality vs. number of views.

All images are kept as **float arrays** (no PNG/8-bit rounding).
The same `metrics.py` should be used by U-Net, NeRF and 3DGS for their evaluation.

## 0. Setup

In [ ]:
# Mount Google Drive (data is NOT in git – too large)
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # get metrics.py from the group repo (branch baseline-sophia)
    if not os.path.exists("/content/IPPR_CT_Reconstruction"):
        !git clone -q -b baseline-sophia https://github.com/gs0232/IPPR_CT_Reconstruction.git /content/IPPR_CT_Reconstruction
    sys.path.insert(0, "/content/IPPR_CT_Reconstruction/Baselines")
else:
    sys.path.insert(0, ".")   # locally: metrics.py lies next to this notebook

In [ ]:
import re, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.transform import iradon

import metrics   # our shared metric functions (PSNR, SSIM, RMSE, head mask)
metrics._self_test()

## 1. Configuration
Adjust the paths once. Everything else can stay as it is.

In [ ]:
DATA_ROOT   = Path("/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction")   # <- adjust if needed
SINO_DIR    = DATA_ROOT / "sinograms"        # <- folder with the sinogram .npy files
FULL_PNG    = DATA_ROOT / "full_output"      # Souram's full-view PNGs (only used for the sanity check)
RESULTS_DIR = DATA_ROOT / "baseline_results" # results + cached ground truth are written here
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_ANGLES    = 984          # projection angles in a full SinoCT sinogram
N_DET       = 888          # detector pixels
ANGLE_RANGE = 360          # degrees covered by the 984 angles (check in section 3!)
OUTPUT_SIZE = 888          # reconstruction size; 888 matches Souram's PNGs
FILTER      = "ramp"       # standard FBP filter

# Number of projection angles kept in the sparse-view scans
VIEW_COUNTS = [246, 120, 60, 30, 15]

SLICE_STEP  = 4            # use every 4th slice of each series (saves time; 1 = all slices)
MAX_SERIES  = 2            # None = all series. Start small to test, then set to None.
SEED        = 42


def view_indices(n_views):
    # Indices of the n_views evenly spaced angles that are kept.
    # Parallel-beam projections at θ and θ+180° contain the same information (mirrored).
    # With a 360° scan we therefore sample only from the first 180°; otherwise e.g.
    # 246 views over 360° would contain only 123 *different* views (every view twice).
    n_half = N_ANGLES if ANGLE_RANGE == 180 else N_ANGLES // 2
    return np.round(np.linspace(0, n_half, n_views, endpoint=False)).astype(int)

for v in VIEW_COUNTS:
    print(f"{v:4d} views -> one view every {180 / v:.1f}°")

## 2. Find and inspect the sinograms

In [ ]:
sino_files = sorted(SINO_DIR.rglob("*.npy"))
print(f"{len(sino_files)} sinogram files found, e.g.:")
for f in sino_files[:5]:
    print("  ", f.relative_to(SINO_DIR))

first = np.load(sino_files[0], mmap_mode="r")   # mmap: only reads what is needed
print("\nshape:", first.shape, "| dtype:", first.dtype,
      "| min/max:", float(np.min(first)), float(np.max(first)))

In [ ]:
def as_slices(arr):
    # Bring any sinogram array into the shape (n_slices, N_DET, N_ANGLES),
    # i.e. one 2D sinogram per slice in the layout that skimage's iradon expects
    # (rows = detector pixels, columns = angles).
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[None]                                   # single slice -> add slice axis
    a_ax = [i for i, s in enumerate(arr.shape) if s == N_ANGLES]
    d_ax = [i for i, s in enumerate(arr.shape) if s == N_DET]
    if not a_ax or not d_ax:
        raise ValueError(f"unexpected sinogram shape {arr.shape}")
    s_ax = ({0, 1, 2} - {a_ax[0], d_ax[0]}).pop()
    return np.transpose(arr, (s_ax, d_ax[0], a_ax[0]))

def series_id(path):
    # 'series_1009.npy' or '.../series_1009/...' -> 'series_1009'
    m = re.search(r"(\d{3,})", str(path.relative_to(SINO_DIR)))
    return f"series_{m.group(1)}" if m else path.stem

demo = as_slices(np.load(sino_files[0]))
print("after reordering:", demo.shape, "(slices, detectors, angles)  |", series_id(sino_files[0]))
plt.figure(figsize=(8, 3))
plt.imshow(demo[demo.shape[0] // 2], cmap="gray", aspect="auto")
plt.xlabel("angle index"); plt.ylabel("detector pixel"); plt.title("sinogram – middle slice")
plt.show()

## 3. FBP function + sanity check against Souram's full-view images
Our full-view FBP should look like Souram's `full_output` PNG for the same slice.
If it looks rotated/mirrored or blurred, the angle range (180° vs 360°) is probably wrong.

In [ ]:
THETA_FULL = np.linspace(0, ANGLE_RANGE, N_ANGLES, endpoint=False)

def fbp(sino, n_views):
    # Sparse-view FBP: keep n_views evenly spaced angles of the full sinogram, then reconstruct.
    # sino: (N_DET, N_ANGLES) float array. n_views = N_ANGLES -> full-view ground truth.
    # Returns (image, runtime in seconds).
    idx = np.arange(N_ANGLES) if n_views == N_ANGLES else view_indices(n_views)
    t0 = time.perf_counter()
    img = iradon(sino[:, idx], theta=THETA_FULL[idx], filter_name=FILTER,
                 output_size=OUTPUT_SIZE, circle=True)
    return img.astype(np.float32), time.perf_counter() - t0

sid = series_id(sino_files[0])
i = demo.shape[0] // 2
gt_demo, t_full = fbp(demo[i], N_ANGLES)
print(f"full-view FBP: {gt_demo.shape}, {t_full:.1f} s per slice")

png = sorted((FULL_PNG / sid).glob("*.png"), key=lambda p: int(re.search(r"(\d+)", p.stem).group(1)))
if len(png) == demo.shape[0]:
    ref = np.array(Image.open(png[i]))[..., 0].astype(np.float32) / 255
    mine = (gt_demo - gt_demo.min()) / (gt_demo.max() - gt_demo.min())
    if ref.shape == mine.shape:
        print("SSIM ours vs. Souram's PNG:", round(metrics.ssim(mine, ref), 3), "(should be close to 1)")
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(mine, cmap="gray"); ax[0].set_title("our full-view FBP")
    ax[1].imshow(ref, cmap="gray");  ax[1].set_title("Souram's full_output PNG")
    for a in ax: a.axis("off")
    plt.show()
else:
    print("no matching PNG series found – compare visually only")
    plt.imshow(gt_demo, cmap="gray"); plt.axis("off"); plt.show()

## 4. One slice at all view counts (figure for the report)

In [ ]:
gt_n = metrics.normalise_pair(gt_demo, gt_demo)[0]
mask = metrics.head_mask(gt_n)

n = len(VIEW_COUNTS)
fig, ax = plt.subplots(2, n + 1, figsize=(3 * (n + 1), 6.4))
ax[0, 0].imshow(gt_n, cmap="gray", vmin=0, vmax=1); ax[0, 0].set_title("ground truth (984 views)")
ax[1, 0].imshow(mask, cmap="gray"); ax[1, 0].set_title("automatic head mask")
for j, v in enumerate(VIEW_COUNTS, start=1):
    rec, _ = fbp(demo[i], v)
    _, rec_n = metrics.normalise_pair(gt_demo, rec)
    m = metrics.evaluate(rec_n, gt_n, already_normalised=True)
    ax[0, j].imshow(rec_n, cmap="gray", vmin=0, vmax=1)
    ax[0, j].set_title(f"FBP {v} views\nPSNR {m['psnr']:.1f} dB, SSIM {m['ssim']:.2f}", fontsize=9)
    ax[1, j].imshow(np.abs(rec_n - gt_n), cmap="magma", vmin=0, vmax=0.3)
    ax[1, j].set_title("|error|", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fbp_views_example.png", dpi=150)
plt.show()

## 5. Main loop: FBP baseline over all series
- The full-view ground truth is cached as `.npy` on Drive, so it is only computed once
  (U-Net / NeRF / 3DGS can use the same files as their ground truth).
- Results are appended to `fbp_baseline.csv` after every series, so a Colab disconnect
  does not lose work – just run the cell again, finished series are skipped.

In [ ]:
GT_DIR = RESULTS_DIR / "gt_fullview_npy"
GT_DIR.mkdir(exist_ok=True)
CSV = RESULTS_DIR / "fbp_baseline.csv"

done = set(pd.read_csv(CSV)["series"]) if CSV.exists() else set()
files = sino_files if MAX_SERIES is None else sino_files[:MAX_SERIES]

for f in files:
    sid = series_id(f)
    if sid in done:
        print(f"{sid}: already done, skipped"); continue
    sinos = as_slices(np.load(f))
    rows, t0 = [], time.perf_counter()
    for s in range(0, sinos.shape[0], SLICE_STEP):
        gt_path = GT_DIR / f"{sid}_slice{s:02d}.npy"
        if gt_path.exists():
            gt = np.load(gt_path)
        else:
            gt, _ = fbp(sinos[s], N_ANGLES)
            np.save(gt_path, gt)
        for v in VIEW_COUNTS:
            rec, rt = fbp(sinos[s], v)
            m = metrics.evaluate(rec, gt)
            rows.append({"series": sid, "slice": s, "method": "FBP", "views": v,
                         "runtime_s": rt, **m})
    pd.DataFrame(rows).to_csv(CSV, mode="a", header=not CSV.exists(), index=False)
    print(f"{sid}: {len(rows) // len(VIEW_COUNTS)} slices in {time.perf_counter() - t0:.0f} s")

## 6. Summary: mean ± SD per view count

In [ ]:
res = pd.read_csv(CSV)
cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head", "runtime_s"]
summary = res.groupby("views")[cols].agg(["mean", "std"]).sort_index(ascending=False).round(3)
summary.to_csv(RESULTS_DIR / "fbp_baseline_summary.csv")
print(f"{res.series.nunique()} series, {len(res) // res.views.nunique()} slices per view count")
summary

In [ ]:
g = res.groupby("views")
x = sorted(res.views.unique())
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, metric, label in [(ax[0], "psnr", "PSNR [dB]"), (ax[1], "ssim", "SSIM")]:
    for col, style, name in [(metric, "o-", "whole image"), (metric + "_head", "s--", "head only")]:
        a.errorbar(x, g[col].mean()[x], yerr=g[col].std()[x], fmt=style, capsize=3, label=name)
    a.set_xscale("log"); a.set_xticks(x); a.set_xticklabels(x); a.xaxis.set_minor_formatter(plt.NullFormatter())
    a.set_xlabel("number of views"); a.set_ylabel(label); a.grid(alpha=0.3); a.legend()
ax[0].set_title("FBP baseline – PSNR vs. views"); ax[1].set_title("FBP baseline – SSIM vs. views")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "fbp_baseline_vs_views.png", dpi=150)
plt.show()

## Notes for the log
- Ground truth = full-view FBP (984 views) of the same sinogram, float, not PNG.
- Sparse views are sampled evenly within 180° (θ and θ+180° are redundant in parallel-beam CT).
  Souram's sparse data and the U-Net/NeRF/3DGS inputs should use the same angles (`view_indices`).
- Both images are scaled with the ground truth's min/max (`metrics.normalise_pair`).
- Head mask: chosen automatically with Otsu's method (no manual threshold).
- Angle range check (section 3): …
- Results: …